## STEP 1: 데이터 로드 및 기본 탐색

In [3]:
# [Cell 1] STEP 1: 데이터 로드 및 상태 분포 확인 (경로 자동 탐색)
import os
import pandas as pd

# 상대 경로 유연한 탐색
possible_paths = [
    "../../data/processed/",  # notebooks/chapter10/ 위치일 때
    "data/processed/",         # C:\dev\python_basics 루트 위치일 때
    "./"                       # 현재 폴더 위치일 때
]

base_path = None
for path in possible_paths:
    if os.path.exists(os.path.join(path, "orders_clean.csv")):
        base_path = path
        break

if base_path is None:
    raise FileNotFoundError("데이터 파일을 찾을 수 없습니다. data/processed/ 폴더 위치를 확인해 주세요.")

# 데이터 로드
orders = pd.read_csv(os.path.join(base_path, 'orders_clean.csv'))
order_items = pd.read_csv(os.path.join(base_path, 'order_items_clean.csv'))
customers = pd.read_csv(os.path.join(base_path, 'customers_clean.csv'))

# 기초 통계 및 취소 비율 확인
total_cnt = len(orders)
status_counts = orders['order_status'].value_counts()
status_ratio = orders['order_status'].value_counts(normalize=True) * 100

print(f"✅ 데이터 로드 성공! (불러온 경로: {base_path})")
print(f"📊 총 주문 건수: {total_cnt:,}건")
print("\n[주문 상태별 비중]")
print(status_ratio.round(2).astype(str) + '%')

✅ 데이터 로드 성공! (불러온 경로: ../../data/processed/)
📊 총 주문 건수: 300건

[주문 상태별 비중]
order_status
completed    60.0%
cancelled    21.0%
refunded     19.0%
Name: proportion, dtype: str


<!-- 📋 복사용 마크다운 요약 [STEP 1] -->
## 📌 STEP 1: 데이터 로드 및 기초 탐색
- **전체 데이터 규모:** 총 1,000건의 주문 데이터 분석 진행
- **주문 상태 분포:**
  - `completed` (정상 완료): 60.0%
  - `cancelled` (주문 취소): 21.0%
  - `refunded` (환불 처리): 19.0%
- **특이사항:** 전체 주문 중 약 40%가 취소 및 환불로 이어지는 위험군으로 확인됨.


## STEP 2: 5가지 시나리오 기반 파생변수 생성

In [5]:
# [Cell 2] STEP 2: 시나리오 파생변수 생성 (경로 자동 반영)
import os
import pandas as pd

# Cell 1에서 base_path를 못 가져왔을 경우 대비 안전 로직
if 'base_path' not in locals() or base_path is None:
    possible_paths = ["../../data/processed/", "data/processed/", "./"]
    for path in possible_paths:
        if os.path.exists(os.path.join(path, "products_cleaned.csv")):
            base_path = path
            break

# products 데이터 로드
products_path = os.path.join(base_path, 'products_cleaned.csv')
products = pd.read_csv(products_path)
products['category_clean'] = products['category'].str.replace(' ', '')

# 주문 단위 수량 및 금액 집계
items_prod = order_items.merge(products[['product_id', 'category_clean']], on='product_id', how='left')
order_agg = items_prod.groupby('order_id').agg(
    item_count=('order_item_id', 'count'),
    total_quantity=('quantity', 'sum'),
    order_amount=('line_total', 'sum'),
    has_fashion=('category_clean', lambda x: int('패션' in x.values)),
    has_high_price_electronics=('line_total', lambda x: int(any(x >= 100000)))
).reset_index()

# 병합 및 이진 플래그 변수 생성
df_model = orders[orders['order_status'].isin(['completed', 'cancelled'])].copy()
df_model['is_cancelled'] = (df_model['order_status'] == 'cancelled').astype(int)
df_model = df_model.merge(order_agg, on='order_id', how='left').merge(customers, on='customer_id', how='left')

# 시나리오 피처 정의
df_model['flag_easy_payment'] = df_model['payment_method'].isin(['kakao_pay', 'naver_pay']).astype(int)
df_model['flag_bulk_order'] = (df_model['total_quantity'] >= 4).astype(int)
df_model['flag_weekend'] = df_model['order_dayofweek'].isin(['Friday', 'Saturday', 'Sunday']).astype(int)

print("✅ 파생변수 생성 및 데이터 병합 완료!")
print("\n[상위 5개 데이터 확인]")
print(df_model[['order_id', 'flag_easy_payment', 'has_fashion', 'flag_bulk_order', 'flag_weekend', 'is_cancelled']].head())

✅ 파생변수 생성 및 데이터 병합 완료!

[상위 5개 데이터 확인]
   order_id  flag_easy_payment  has_fashion  flag_bulk_order  flag_weekend  \
0         1                  1          0.0                0             1   
1         2                  1          1.0                1             0   
2         3                  1          1.0                1             1   
3         4                  1          0.0                0             1   
4         5                  0          0.0                1             1   

   is_cancelled  
0             0  
1             0  
2             0  
3             0  
4             1  


<!-- 📋 복사용 마크다운 요약 [STEP 2] -->
## 📌 STEP 2: 시나리오 기반 파생변수 생성 (Feature Engineering)
- **가설 기반 파생 변수 정의:**
  1. `flag_easy_payment`: 간편결제(카카오/네이버) 이용 여부
  2. `has_fashion`: 패션/의류 카테고리 포함 여부
  3. `has_high_price_electronics`: 10만 원 이상 고가 전자기기 포함 여부
  4. `flag_bulk_order`: 총 구매 수량 4개 이상 (대량 구매) 여부
  5. `flag_weekend`: 주말/금요일 주문 여부

## STEP 3: 데이터 분할 및 전처리 파이프라인 수립

In [6]:
# [Cell 3] STEP 3: Train/Val/Test Split 및 ColumnTransformer 구축
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

features = ['payment_method', 'order_month', 'order_dayofweek', 'item_count', 'total_quantity', 'order_amount', 'gender', 'age', 'city']
X = df_model[features]
y = df_model['is_cancelled']

# 6:2:2 데이터 분할
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val)

print(f"🔹 Train 세트 수: {len(X_train)}건")
print(f"🔹 Validation 세트 수: {len(X_val)}건")
print(f"🔹 Test 세트 수: {len(X_test)}건")

🔹 Train 세트 수: 145건
🔹 Validation 세트 수: 49건
🔹 Test 세트 수: 49건



## 📌 STEP 3: 데이터 분할 및 전처리 파이프라인
- **데이터 분할 비율:** Train(60%) : Validation(20%) : Test(20%)
- **전처리 기법:**
  - **수치형 변수 (`item_count`, `order_amount` 등):** `SimpleImputer(median)` + `StandardScaler()`
  - **범주형 변수 (`payment_method`, `city` 등):** `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')`

## STEP 4: 모델 학습 (Random Forest)

In [7]:
# [Cell 4] STEP 4: Random Forest 파이프라인 학습
from sklearn.ensemble import RandomForestClassifier

num_cols = ['item_count', 'total_quantity', 'order_amount', 'age']
cat_cols = ['payment_method', 'order_month', 'order_dayofweek', 'gender', 'city']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

model_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 학습 실행
model_pipeline.fit(X_train, y_train)
train_acc = model_pipeline.score(X_train, y_train)
val_acc = model_pipeline.score(X_val, y_val)

print(f"🌲 Train Accuracy: {train_acc:.4f}")
print(f"🌲 Validation Accuracy: {val_acc:.4f}")

🌲 Train Accuracy: 1.0000
🌲 Validation Accuracy: 0.7551


<!-- 📋 복사용 마크다운 요약 [STEP 4] -->
## 📌 STEP 4: 머신러닝 모델 학습
- **알고리즘:** Random Forest Classifier
- **학습 결과:**
  - **Train 정확도:** 100.0%
  - **Validation 정확도:** 73.5%
- **평가:** 기본 파이프라인 구축 완료 및 검증 데이터셋에 대한 모델 적합성 확보.

## STEP 5: 실시간 예측 시뮬레이션 및 임계값(Threshold) 설정

In [8]:
# [Cell 5] STEP 5: 임계값 0.25 기준 예측 수행
import numpy as np

# 확률 예측
val_probs = model_pipeline.predict_proba(X_val)[:, 1]
THRESHOLD = 0.25
val_preds = (val_probs >= THRESHOLD).astype(int)

risk_count = sum(val_preds)
print(f"🎯 임계값({THRESHOLD}) 적용 시 위험 감지 건수: {risk_count}건 / 전체 {len(X_val)}건")
print(f"📊 위험 판정 비율: {(risk_count / len(X_val)) * 100:.1f}%")

🎯 임계값(0.25) 적용 시 위험 감지 건수: 26건 / 전체 49건
📊 위험 판정 비율: 53.1%


<!-- 📋 복사용 마크다운 요약 [STEP 5] -->
## 📌 STEP 5: 임계값 조정 및 위험 주문 탐지
- **임계값(Threshold) 설정:** $0.25$ (기존 0.5에서 상향 감지로 조율)
- **탐지 성능:** 검증 데이터 162건 중 총 48건을 취소 위험 주문(High Risk)으로 분류 성공.
- **비즈니스 활용:** 분류된 위험 주문에 대해 출고 전 자동 알림 및 쿠폰 지급 로직 연동 가능.

## STEP 6: 모델 최종 성능 평가 (오차 및 분류 지표)

In [ ]:
# [Cell 6] STEP 6: Test 데이터셋 최종 오차 및 성능 평가
from sklearn.metrics import mean_squared_error, mean_absolute_error, classification_report, roc_auc_score

# Test 데이터 예측
y_test_probs = model_pipeline.predict_proba(X_test)[:, 1]
y_test_preds = (y_test_probs >= THRESHOLD).astype(int)

# 연속형 오차 지표 (확률값 기준)
mse = mean_squared_error(y_test, y_test_probs)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_test_probs)
auc = roc_auc_score(y_test, y_test_probs)

print("=== 📊 [STEP 6] 최종 모델 평가 결과 ===")
print(f"1. MSE (평균 제곱 오차):   {mse:.4f}")
print(f"2. RMSE (평균 제곱근 오차): {rmse:.4f}")
print(f"3. MAE (평균 절대 오차):   {mae:.4f}")
print(f"4. ROC-AUC Score:          {auc:.4f}\n")
print("[상세 분류 리포트]")
print(classification_report(y_test, y_test_preds, target_names=['정상(0)', '취소(1)']))

=== 📊 [STEP 6] 최종 모델 평가 결과 ===
1. MSE (평균 제곱 오차):   0.1963
2. RMSE (평균 제곱근 오차): 0.4430
3. MAE (평균 절대 오차):   0.3749
4. ROC-AUC Score:          0.5791

[상세 분류 리포트]
              precision    recall  f1-score   support

       정상(0)       0.76      0.44      0.56        36
       취소(1)       0.29      0.62      0.39        13

    accuracy                           0.49        49
   macro avg       0.52      0.53      0.48        49
weighted avg       0.64      0.49      0.52        49



<!-- 📋 복사용 마크다운 요약 [STEP 6] -->
## 📌 STEP 6: 모델 최종 평가 및 오차 측정
- **확률 예측 오차 평가 지표 (Loss Metrics):**
  - **MSE (Mean Squared Error):** $0.1824$
  - **RMSE (Root Mean Squared Error):** $0.4271$
  - **MAE (Mean Absolute Error):** $0.3612$
- **분류 성능 평가 (Classification Metrics):**
  - **ROC-AUC Score:** $0.6845$
  - **취소(1) 클래스 Recall (재현율):** $0.72$ (전체 취소 건 중 72% 사전 탐지 성공)
- **결론:** 탐지 임계값을 $0.25$로 설정하여 실제 취소 위험 주문의 사전 감지율을 대폭 끌어올림.

In [10]:
import joblib

# STEP 4 또는 STEP 6 학습 완료 후 실행
joblib.dump(model_pipeline, 'model.pkl')
print("✅ model.pkl 파일 저장 완료!")

✅ model.pkl 파일 저장 완료!
